In [1]:
import sys
import logging
from pathlib import Path
import pandas as pd
import json
from tqdm.auto import tqdm
import sqlite3

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from src.data.database import EntityDatabase
from src.data.question_generator import MultipleChoiceQuestionGenerator

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

/home/khaledbouabdallah/repos/llm_grouping_loss_INRIA/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'src.data.question_generator'

In [2]:
db_path = Path('../data/entities.db')
conn = sqlite3.connect(db_path)

print(f"✓ Connected to database: {db_path}")
print(f"  Database size: {db_path.stat().st_size / 1024 / 1024:.2f} MB")

✓ Connected to database: ../data/entities.db
  Database size: 806.34 MB


## 1. Setup

In [3]:
# Initialize database
db = EntityDatabase(db_path)

print(f"Database initialized: {db.db_path}")

2026-02-02 12:08:56,688 - src.data.database - INFO - Database initialized at ../data/entities.db


Database initialized: ../data/entities.db


## 2. Load Selected Entities

In [20]:
# Load selected entities from notebook 01
selected_entities_path = Path('../data/experiment/selected_entities.csv')

if not selected_entities_path.exists():
    raise FileNotFoundError(
        f"Selected entities file not found: {selected_entities_path}\n"
        "Please run notebook 01_select_experiment_data.ipynb first."
    )

entities_df = pd.read_csv(selected_entities_path)

print(f"Loaded {len(entities_df)} entities")
print(f"\nCulture distribution:")
print(entities_df['culture'].value_counts())
print(f"\nCountry distribution (top 10):")
print(entities_df['country'].value_counts().head(10))

Loaded 23007 entities

Culture distribution:
culture
Western            6010
Latin American     5150
East Asian         4096
Arabic             3714
African            2329
South Asian        1247
Southeast Asian     461
Name: count, dtype: int64

Country distribution (top 10):
country
Japan             1666
United States     1650
Brazil            1589
Morocco           1514
Germany           1487
France            1480
Argentina         1451
United Kingdom    1393
Egypt             1275
Mexico            1215
Name: count, dtype: int64


## 3. Build Entity Pool for Distractor Selection

In [21]:
# Get entity pool: map Wikidata IDs to database IDs
# This restricts distractor selection to only our selected entities
print("Building entity pool for distractor selection...")

entity_pool = []
failed_lookups = 0

for wikidata_id in entities_df['wikidata_id']:
    entity = db.get_entity(wikidata_id)
    if entity:
        entity_pool.append(entity['id'])
    else:
        failed_lookups += 1

print(f"✓ Entity pool: {len(entity_pool)} database IDs")
if failed_lookups > 0:
    print(f"⚠ Warning: {failed_lookups} entities not found in database")

# Initialize question generator WITH entity pool restriction
generator = MultipleChoiceQuestionGenerator(db, entity_pool=entity_pool)

print(f"✓ Question generator initialized with entity pool restriction")
print(f"  Distractors will only be selected from the {len(entity_pool)} selected entities")

Building entity pool for distractor selection...
✓ Entity pool: 23007 database IDs
✓ Question generator initialized with entity pool restriction
  Distractors will only be selected from the 23007 selected entities


## 5. Generate Questions for All Entities

In [22]:
# Generate questions for all selected entities
total_stats = {"success": 0, "failed": 0, "skipped": 0}
entity_results = []

print(f"Generating questions for {len(entity_pool)} entities...")
print(f"Expected: {len(entity_pool) * 9} questions (3 types × 3 languages)\n")

for i, entity_id in enumerate(tqdm(entity_pool, desc="Generating questions")):
    try:
        # Get entity info from DataFrame for reporting
        entity_row = entities_df[entities_df['wikidata_id'] == db.get_entity_by_id(entity_id)['wikidata_id']].iloc[0]

        # Generate all questions for this entity
        stats = generator.generate_all_questions(entity_id)

        # Update totals
        for key in total_stats:
            total_stats[key] += stats[key]

        # Store result
        entity_results.append({
            'entity_id': entity_id,
            'wikidata_id': entity_row['wikidata_id'],
            'culture': entity_row['culture'],
            'country': entity_row['country'],
            'success': stats['success'],
            'failed': stats['failed'],
            'skipped': stats['skipped']
        })

    except Exception as e:
        logger.error(f"Failed to generate questions for entity {entity_id}: {e}")
        entity_results.append({
            'entity_id': entity_id,
            'wikidata_id': 'Unknown',
            'culture': 'Unknown',
            'country': 'Unknown',
            'success': 0,
            'failed': 9,
            'skipped': 0
        })
        total_stats['failed'] += 9

print("\n" + "="*60)
print("GENERATION COMPLETE")
print("="*60)
print(f"Success: {total_stats['success']} questions")
print(f"Failed:  {total_stats['failed']} questions")
print(f"Skipped: {total_stats['skipped']} questions (already existed)")
print(f"Total:   {sum(total_stats.values())} operations")

success_rate = (total_stats['success'] / (len(entity_pool) * 9) * 100) if entity_pool else 0
print(f"\nSuccess rate: {success_rate:.1f}%")

Generating questions for 23007 entities...
Expected: 207063 questions (3 types × 3 languages)



Generating questions:   0%|          | 0/23007 [00:00<?, ?it/s]

2026-01-07 01:29:03,386 - src.data.question_generator - INFO - Generating questions for entity 1 (Q10738)
2026-01-07 01:29:03,505 - src.data.question_generator - INFO - Question generation complete: {'success': 9, 'failed': 0, 'skipped': 0}
2026-01-07 01:29:03,506 - src.data.question_generator - INFO - Generating questions for entity 2 (Q10731)
2026-01-07 01:29:03,598 - src.data.question_generator - INFO - Question generation complete: {'success': 9, 'failed': 0, 'skipped': 0}
2026-01-07 01:29:03,599 - src.data.question_generator - INFO - Generating questions for entity 3 (Q1124)
2026-01-07 01:29:03,690 - src.data.question_generator - INFO - Question generation complete: {'success': 9, 'failed': 0, 'skipped': 0}
2026-01-07 01:29:03,691 - src.data.question_generator - INFO - Generating questions for entity 4 (Q1744)
2026-01-07 01:29:03,781 - src.data.question_generator - INFO - Question generation complete: {'success': 9, 'failed': 0, 'skipped': 0}
2026-01-07 01:29:03,783 - src.data.que


GENERATION COMPLETE
Success: 205098 questions
Failed:  1965 questions
Skipped: 0 questions (already existed)
Total:   207063 operations

Success rate: 99.1%


## 6. Analyze Generation Results

In [31]:
# Analyze entity-level results
results_df = pd.DataFrame(entity_results)

# Entities with all 9 questions generated
complete_entities = results_df[results_df['success'] == 9]
partial_entities = results_df[(results_df['success'] > 0) & (results_df['success'] < 9)]
failed_entities = results_df[results_df['success'] == 0]

print(f"Entities with all 9 questions:  {len(complete_entities)} ({len(complete_entities)/len(results_df)*100:.1f}%)")
print(f"Entities with partial coverage: {len(partial_entities)} ({len(partial_entities)/len(results_df)*100:.1f}%)")
print(f"Entities with no questions:     {len(failed_entities)} ({len(failed_entities)/len(results_df)*100:.1f}%)")

if len(failed_entities) > 0:
    print(f"\nFailed entities by culture:")
    print(failed_entities['culture'].value_counts())

if len(partial_entities) > 0:
    print(f"\nPartial coverage entities:")
    print(partial_entities[['entity_id', 'wikidata_id', 'culture', 'success', 'failed']].head(10))

Entities with all 9 questions:  22352 (97.2%)
Entities with partial coverage: 655 (2.8%)
Entities with no questions:     0 (0.0%)

Partial coverage entities:
       entity_id wikidata_id      culture  success  failed
13823      21709       Q1047  South Asian        6       3
13826      21714      Q92830  South Asian        6       3
13827      21715       Q7406  South Asian        6       3
13830      21719     Q103774  South Asian        6       3
13831      21720      Q60429  South Asian        6       3
13833      21725      Q80309  South Asian        6       3
13835      21727      Q81576  South Asian        6       3
13836      21728      Q45789  South Asian        6       3
13837      21729     Q173118  South Asian        6       3
13838      21730     Q310773  South Asian        6       3


## 6. Validate Question Database

In [24]:
# Load all questions from database
all_questions = db.get_all_questions()
questions_df = pd.DataFrame(all_questions)

print(f"Total questions in database: {len(questions_df)}")
print(f"Unique entities: {questions_df['entity_id'].nunique()}")

print(f"\nLanguage distribution:")
lang_counts = questions_df['language'].value_counts()
for lang in ['en', 'fr', 'ar']:
    count = lang_counts.get(lang, 0)
    print(f"  {lang}: {count} ({count/len(questions_df)*100:.1f}%)")

print(f"\nQuestion type distribution:")
type_counts = questions_df['question_type'].value_counts()
for qtype in ['birth_place', 'nationality', 'birth_date']:
    count = type_counts.get(qtype, 0)
    print(f"  {qtype}: {count} ({count/len(questions_df)*100:.1f}%)")

print(f"\nQuestions per entity (should be 9):")
questions_per_entity = questions_df.groupby('entity_id').size()
print(questions_per_entity.value_counts().sort_index())

Total questions in database: 205098
Unique entities: 23007

Language distribution:
  en: 68366 (33.3%)
  fr: 68366 (33.3%)
  ar: 68366 (33.3%)

Question type distribution:
  birth_place: 69021 (33.7%)
  nationality: 67056 (32.7%)
  birth_date: 69021 (33.7%)

Questions per entity (should be 9):
6      655
9    22352
Name: count, dtype: int64


## 6.1. Diagnose Missing Questions

In [25]:
# Identify incomplete entities
questions_per_entity = questions_df.groupby('entity_id').size()
incomplete_entities = questions_per_entity[questions_per_entity < 9]

print(f"Entities with incomplete questions: {len(incomplete_entities)}")
print(f"\nBreakdown:")
for count in sorted(incomplete_entities.unique()):
    num_entities = (incomplete_entities == count).sum()
    missing = 9 - count
    print(f"  {num_entities} entities have {count} questions (missing {missing})")

print(f"\nTotal missing questions: {len(incomplete_entities) * 9 - incomplete_entities.sum()}")

# Get the incomplete entity IDs
incomplete_entity_ids = incomplete_entities.index.tolist()
print(f"\nSample incomplete entity IDs: {incomplete_entity_ids[:10]}")

Entities with incomplete questions: 655

Breakdown:
  655 entities have 6 questions (missing 3)

Total missing questions: 1965

Sample incomplete entity IDs: [21709, 21714, 21715, 21719, 21720, 21725, 21727, 21728, 21729, 21730]


In [26]:
# Analyze what types/languages are missing for incomplete entities
incomplete_questions = questions_df[questions_df['entity_id'].isin(incomplete_entity_ids)]

# Expected combinations: 3 types × 3 languages = 9
expected_combinations = [
    (qtype, lang)
    for qtype in ['birth_place', 'nationality', 'birth_date']
    for lang in ['en', 'fr', 'ar']
]

# Check which combinations are missing for a sample of incomplete entities
print("Analyzing missing question combinations for incomplete entities...")
print(f"\nSample entity analysis (showing first 10):\n")

sample_incomplete = incomplete_entity_ids[:10]

for entity_id in sample_incomplete:
    entity_questions = questions_df[questions_df['entity_id'] == entity_id]
    actual_combinations = set(zip(entity_questions['question_type'], entity_questions['language']))
    missing_combinations = set(expected_combinations) - actual_combinations

    print(f"Entity {entity_id}: {len(entity_questions)}/9 questions")
    if missing_combinations:
        for qtype, lang in sorted(missing_combinations):
            print(f"  ✗ Missing: {qtype} in {lang}")
    print()

Analyzing missing question combinations for incomplete entities...

Sample entity analysis (showing first 10):

Entity 21709: 6/9 questions
  ✗ Missing: nationality in ar
  ✗ Missing: nationality in en
  ✗ Missing: nationality in fr

Entity 21714: 6/9 questions
  ✗ Missing: nationality in ar
  ✗ Missing: nationality in en
  ✗ Missing: nationality in fr

Entity 21715: 6/9 questions
  ✗ Missing: nationality in ar
  ✗ Missing: nationality in en
  ✗ Missing: nationality in fr

Entity 21719: 6/9 questions
  ✗ Missing: nationality in ar
  ✗ Missing: nationality in en
  ✗ Missing: nationality in fr

Entity 21720: 6/9 questions
  ✗ Missing: nationality in ar
  ✗ Missing: nationality in en
  ✗ Missing: nationality in fr

Entity 21725: 6/9 questions
  ✗ Missing: nationality in ar
  ✗ Missing: nationality in en
  ✗ Missing: nationality in fr

Entity 21727: 6/9 questions
  ✗ Missing: nationality in ar
  ✗ Missing: nationality in en
  ✗ Missing: nationality in fr

Entity 21728: 6/9 questions
  ✗ Mi

In [27]:
# Aggregate missing question patterns across all incomplete entities
from collections import Counter

missing_patterns = Counter()

for entity_id in incomplete_entity_ids:
    entity_questions = questions_df[questions_df['entity_id'] == entity_id]
    actual_combinations = set(zip(entity_questions['question_type'], entity_questions['language']))
    missing_combinations = set(expected_combinations) - actual_combinations

    for qtype, lang in missing_combinations:
        missing_patterns[(qtype, lang)] += 1

print("Most common missing question combinations:")
print(f"{'Question Type':<15} {'Language':<10} {'Count':<10} {'% of incomplete'}")
print("-" * 60)
for (qtype, lang), count in missing_patterns.most_common():
    pct = count / len(incomplete_entity_ids) * 100
    print(f"{qtype:<15} {lang:<10} {count:<10} {pct:.1f}%")

Most common missing question combinations:
Question Type   Language   Count      % of incomplete
------------------------------------------------------------
nationality     en         655        100.0%
nationality     ar         655        100.0%
nationality     fr         655        100.0%


In [28]:
# Check if incomplete entities have specific characteristics
# First, check what cultures these entities actually have vs what we expect

# The issue: entities from India/Pakistan/Bangladesh should have culture "South Asian"
# but the fallback dict uses "South Asian" while entities might be labeled differently

print("Checking culture labels of incomplete entities...\n")

incomplete_entity_db_ids = incomplete_entity_ids

# Get entity info from database
incomplete_entity_info = []
for db_id in incomplete_entity_db_ids:
    entity = db.get_entity_by_id(db_id)
    if entity:
        incomplete_entity_info.append(entity)

incomplete_df = pd.DataFrame(incomplete_entity_info)

print(f"Sample of incomplete entities with their culture labels:")
if len(incomplete_df) > 0 and 'country' in incomplete_df.columns and 'culture' in incomplete_df.columns:
    sample = incomplete_df[['wikidata_id', 'country', 'culture']].head(20)
    print(sample.to_string(index=False))
else:
    print("  ⚠ No culture or country columns found!")
    print(f"  Available columns: {list(incomplete_df.columns)}")
print()

print(f"Incomplete entities metadata:")
print(f"Total: {len(incomplete_df)}")

if 'culture' in incomplete_df.columns:
    print(f"\nBy culture:")
    culture_incomplete = incomplete_df['culture'].value_counts()
    for culture in ['Western', 'Arab', 'African']:
        count = culture_incomplete.get(culture, 0)
        total_culture = len(entities_df[entities_df['culture'] == culture])
        pct = count / total_culture * 100 if total_culture > 0 else 0
        print(f"  {culture:10s}: {count:5d} / {total_culture:5d} ({pct:.1f}%)")

if 'country' in incomplete_df.columns:
    print(f"\nTop 10 countries with incomplete entities:")
    print(incomplete_df['country'].value_counts().head(10))

Checking culture labels of incomplete entities...

Sample of incomplete entities with their culture labels:
wikidata_id country     culture
      Q1047   India South Asian
     Q92830   India South Asian
      Q7406   India South Asian
    Q103774   India South Asian
     Q60429   India South Asian
     Q80309   India South Asian
     Q81576   India South Asian
     Q45789   India South Asian
    Q173118   India South Asian
    Q310773   India South Asian
    Q277422   India South Asian
    Q315338   India South Asian
      Q9488   India South Asian
    Q310867   India South Asian
    Q144385   India South Asian
     Q10218   India South Asian
    Q333460   India South Asian
    Q233289   India South Asian
    Q313515   India South Asian
    Q156347   India South Asian

Incomplete entities metadata:
Total: 655

By culture:
  Western   :     0 /  6010 (0.0%)
  Arab      :     0 /     0 (0.0%)
  African   :     0 /  2329 (0.0%)

Top 10 countries with incomplete entities:
country
India   

In [29]:
# Check the actual entity data to see if missing questions are due to missing data
# Let's examine a few incomplete entities in detail

print("Detailed analysis of sample incomplete entities:\n")

sample_incomplete_entities = incomplete_entity_ids[:5]

for entity_id in sample_incomplete_entities:
    entity = db.get_entity_by_id(entity_id)
    if entity:
        print(f"{'='*80}")
        print(f"Entity ID: {entity_id} | Wikidata ID: {entity['wikidata_id']}")
        print(f"Name: {entity.get('label', entity.get('name', 'N/A'))} | Country: {entity.get('country', 'N/A')} | Culture: {entity.get('culture', 'N/A')}")

        # Check what data is available
        print(f"\nAvailable data:")
        print(f"  Birth place: {entity.get('birth_place', 'MISSING')}")
        print(f"  Birth country: {entity.get('birth_country', 'MISSING')}")
        print(f"  Birth date: {entity.get('birth_date', 'MISSING')}")
        print(f"  Nationality: {entity.get('nationality', 'MISSING')}")
        print(f"  Labels: en={entity.get('label_en', 'N/A')}, fr={entity.get('label_fr', 'N/A')}, ar={entity.get('label_ar', 'N/A')}")

        # Show which questions exist
        entity_questions = questions_df[questions_df['entity_id'] == entity_id]
        print(f"\nExisting questions ({len(entity_questions)}/9):")
        for _, q in entity_questions.iterrows():
            print(f"  ✓ {q['question_type']:<15} {q['language']}")

        # Show which are missing
        actual_combinations = set(zip(entity_questions['question_type'], entity_questions['language']))
        missing_combinations = set(expected_combinations) - actual_combinations
        if missing_combinations:
            print(f"\nMissing questions ({len(missing_combinations)}):")
            for qtype, lang in sorted(missing_combinations):
                print(f"  ✗ {qtype:<15} {lang}")
        print()

Detailed analysis of sample incomplete entities:

Entity ID: 21709 | Wikidata ID: Q1047
Name: N/A | Country: India | Culture: South Asian

Available data:
  Birth place: MISSING
  Birth country: MISSING
  Birth date: MISSING
  Nationality: MISSING
  Labels: en=N/A, fr=N/A, ar=N/A

Existing questions (6/9):
  ✓ birth_place     en
  ✓ birth_place     fr
  ✓ birth_place     ar
  ✓ birth_date      en
  ✓ birth_date      fr
  ✓ birth_date      ar

Missing questions (3):
  ✗ nationality     ar
  ✗ nationality     en
  ✗ nationality     fr

Entity ID: 21714 | Wikidata ID: Q92830
Name: N/A | Country: India | Culture: South Asian

Available data:
  Birth place: MISSING
  Birth country: MISSING
  Birth date: MISSING
  Nationality: MISSING
  Labels: en=N/A, fr=N/A, ar=N/A

Existing questions (6/9):
  ✓ birth_place     en
  ✓ birth_place     fr
  ✓ birth_place     ar
  ✓ birth_date      en
  ✓ birth_date      fr
  ✓ birth_date      ar

Missing questions (3):
  ✗ nationality     ar
  ✗ nationality 

## 6.2. Entity 19738 - Multiple Birth Places Issue

**Issue Found:** Entity 19738 (Q380118 - South Korean singer) has 2 birth places:
- **Gangdong District** (Q50348) - only has EN & FR labels, missing AR
- **Seoul** (Q8684) - has all 3 languages ✓

**Why it passed validation:** Notebook 01 only checked if *at least one* fact had complete labels, not that *ALL* facts of each property had complete labels.

**Fix Applied:** Updated notebook 01 validation to check ALL fact values have labels in all languages before including an entity in the experiment.

**Action Taken:** 
1. ✓ Updated validation logic in notebook 01
2. ✓ Deleted all 207,071 questions from database
3. Next: Re-run notebook 01 with stricter validation, then regenerate all questions

## 7. Analyze Balance by Culture

## 8. Show Sample Questions

In [30]:
# Sample questions from each type and language
import random

def display_question(question_row):
    """Pretty print a question"""
    print(f"\n{'='*80}")
    print(f"Question ID: {question_row['id']}")
    print(f"Entity: {question_row['wikidata_id']} (Culture: {question_row['culture']}, Country: {question_row['country']})")
    print(f"Type: {question_row['question_type']} | Language: {question_row['language']}")
    print(f"\n{question_row['question_text']}")

    # Parse options
    options = json.loads(question_row['options'])
    for opt in options:
        marker = "✓" if opt['letter'] == question_row['correct_answer'] else " "
        print(f"  {marker} {opt['letter']}. {opt['text']}")

    print(f"\nGround truth: {question_row['ground_truth']}")
    print(f"Correct answer: {question_row['correct_answer']}")
    print(f"{'='*80}")

# Show 2 examples from each question type
for qtype in ['birth_place', 'nationality', 'birth_date']:
    print(f"\n\n{'#'*80}")
    print(f"# SAMPLE QUESTIONS: {qtype.upper()}")
    print(f"{'#'*80}")

    # Get questions of this type
    type_questions = questions_with_meta[questions_with_meta['question_type'] == qtype]

    # Sample 2 different entities
    if len(type_questions) > 0:
        sample_entities = random.sample(
            list(type_questions['entity_id'].unique()),
            min(2, len(type_questions['entity_id'].unique()))
        )

        for entity_id in sample_entities:
            # Show all 3 languages for this entity
            entity_questions = type_questions[type_questions['entity_id'] == entity_id]
            for _, question in entity_questions.iterrows():
                display_question(question)



################################################################################
# SAMPLE QUESTIONS: BIRTH_PLACE
################################################################################



KeyError: 'id'

## 9. Export Questions for LLM Testing

In [ ]:
# Export questions to CSV for LLM evaluation
output_path = Path('../data/experiment/questions_for_llm.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

# Prepare export dataframe
export_df = questions_with_meta[[
    'id', 'entity_id', 'wikidata_id', 'culture', 'country',
    'language', 'question_type', 'question_text',
    'options', 'correct_answer', 'ground_truth', 'property_code'
]].copy()

export_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Exported {len(export_df)} questions to: {output_path}")

# Also export as JSONL for easier LLM consumption
jsonl_path = output_path.with_suffix('.jsonl')
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for _, row in export_df.iterrows():
        record = row.to_dict()
        # Parse options from JSON string
        record['options'] = json.loads(record['options'])
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"Exported questions to JSONL: {jsonl_path}")

## 10. Summary Statistics

In [ ]:
print("="*80)
print("QUESTION GENERATION SUMMARY")
print("="*80)

print(f"\nSelected entities: {len(entities_df)}")
print(f"Total questions generated: {len(questions_df)}")
print(f"Expected questions (9 per entity): {len(entities_df) * 9}")
print(f"Coverage: {len(questions_df)/(len(entities_df)*9)*100:.1f}%")

print(f"\nQuestions by culture:")
for culture in ['Western', 'Arab', 'African']:
    count = len(questions_with_meta[questions_with_meta['culture'] == culture])
    entities = len(entities_df[entities_df['culture'] == culture])
    expected = entities * 9
    print(f"  {culture:10s}: {count:5d} / {expected:5d} ({count/expected*100:.1f}%)")

print(f"\nQuestions by language:")
for lang in ['en', 'fr', 'ar']:
    count = len(questions_df[questions_df['language'] == lang])
    expected = len(entities_df) * 3  # 3 question types per language
    print(f"  {lang}: {count:5d} / {expected:5d} ({count/expected*100:.1f}%)")

print(f"\nQuestions by type:")
for qtype in ['birth_place', 'nationality', 'birth_date']:
    count = len(questions_df[questions_df['question_type'] == qtype])
    expected = len(entities_df) * 3  # 3 languages per type
    print(f"  {qtype:15s}: {count:5d} / {expected:5d} ({count/expected*100:.1f}%)")

print(f"\nFiles exported:")
print(f"  - {output_path}")
print(f"  - {jsonl_path}")

print("\n" + "="*80)
print("READY FOR LLM EVALUATION (Notebook 05)")
print("="*80)